In [ ]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if not os.getcwd().endswith(target_folder):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

This notebook shows an example of using modules/ to train a model. If you are going to use this to create a new trained model make sure to change the model name variable to ensure you dont overwrite existing models

In [ ]:
import sys
import os
from pathlib import Path

root_path = Path.cwd().parent
os.chdir(root_path)  # change cwd to project root

if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from modules import ImagenetLoader, datasetPrepper, modelTrainer
import pandas as pd

pretrained_weights_path = "RadImageNet_weights/resnet50.pth"
dataframe_path = "data/labels.csv"
image_dir = "data/test_images"
model_name = "resnet50_baseline_cpu"

dataframe = pd.read_csv(dataframe_path)
num_classes = dataframe["label"].nunique()

dataframe.head()

c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,filename,pathology,modality,location,label
0,test_0.png,soft_tissue_fluid,mri,ankle foot,ankle_foot_soft_tissue_fluid
1,test_1.png,osseous_disruption,mri,ankle foot,ankle_foot_osseous_disruption
2,test_2.png,chondral_abnormality,mri,ankle foot,ankle_foot_chondral_abnormality
3,test_3.png,achilles_pathology_,mri,ankle foot,ankle_foot_achilles_pathology_
4,test_4.png,achilles_pathology_,mri,ankle foot,ankle_foot_achilles_pathology_


In [2]:
import os

# should be the root of the project
print(os.getcwd())

c:\Users\fiach\Documents\Code\msc\scalable\CS6423_knowledge_distillation_project


In [3]:
loader = ImagenetLoader()
loader.load_radimagenet_resnet50(pretrained_weights_path)
loader.freeze_backbone()

model = loader.model
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [4]:
data = datasetPrepper(
    dataframe_path="data/labels.csv",
    image_dir="data/test_images",
).prepare(compute_class_weights=True)

print(f"Dataset prepared:")
print(f"  Train samples: {len(data.train_dataset)}")
print(f"  Val samples: {len(data.val_dataset)}")
print(f"  Test samples: {len(data.test_dataset)}")
print(f"  Classes: {len(data.class_names)}")

Dataset prepared:
  Train samples: 7948
  Val samples: 1590
  Test samples: 398
  Classes: 61


In [5]:
trainer = modelTrainer(
    model=model,
    data_prep=data,
    device=None,
    learn_rate=0.001,
    num_epochs=10,
    model_name=model_name,
)

trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
print("Ready to train")

Ready to train


In [6]:
trainer.train_all()

Epoch 1/10:   0%|          | 0/249 [00:00<?, ?batch/s]c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validating: 100%|██████████| 50/50 [02:10<00:00,  2.60s/batch]



Epoch 1/10
Train Loss: 1.5258 | Train F1: 0.3555
Val Loss: 1.4834 | Val F1: 0.3808
Epoch Time: 876.21s



Epoch 2/10:   0%|          | 0/249 [00:00<?, ?batch/s]c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validating: 100%|██████████| 50/50 [02:07<00:00,  2.54s/batch]



Epoch 2/10
Train Loss: 0.7735 | Train F1: 0.5494
Val Loss: 1.3347 | Val F1: 0.4134
Epoch Time: 858.46s



Epoch 3/10:   0%|          | 0/249 [00:00<?, ?batch/s]c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validating: 100%|██████████| 50/50 [02:10<00:00,  2.61s/batch]


...


Epoch 8/10
Train Loss: 0.4658 | Train F1: 0.6613
Val Loss: 1.2639 | Val F1: 0.4596
Epoch Time: 857.31s



Epoch 9/10:   0%|          | 0/249 [00:00<?, ?batch/s]c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validating: 100%|██████████| 50/50 [02:09<00:00,  2.58s/batch]



Epoch 9/10
Train Loss: 0.4499 | Train F1: 0.6685
Val Loss: 1.2535 | Val F1: 0.4712
Epoch Time: 860.85s



Epoch 10/10:   0%|          | 0/249 [00:00<?, ?batch/s]c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validating: 100%|██████████| 50/50 [02:08<00:00,  2.57s/batch]


Epoch 10/10
Train Loss: 0.4267 | Train F1: 0.6881
Val Loss: 1.3700 | Val F1: 0.4544
Epoch Time: 861.70s

